# 02 · El embudo de comparabilidad

**Objetivo:** Mostrar a directores, dueños y equipos comerciales que una diferencia de precio entre municipios no basta para hablar de una oportunidad de rentabilidad, y cuánto de esa diferencia sobrevive cuando se valida que los productos, las presentaciones y los periodos son comparables.

**Pregunta:** ¿Cuánto de esa diferencia desaparece al validar la comparabilidad, paso por paso?

Por qué importa: si la diferencia baja mucho al exigir que se compare lo mismo, la cifra ingenua no sirve
para decidir. Este notebook mide cuánto baja en cada paso y cuánto de lo que queda es más que azar.

**Decisión:** universo fijo (los mismos 280 nombres en cada paso) e IC90 con los mismos municipios sorteados en cada
paso, para que el intervalo de «cuánto desaparece» sea coherente. Alternativa descartada: recalcular el universo en
cada paso (la diferencia cambiaría solo porque cambian los productos).

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import embudo as M
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
def reg(*a, **k):
    entorno.registrar(*a, **k)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
raw = entorno.load_raw()
d_all = M.prep(raw)
d = d_all[~d_all.catalogo.isin(M.FUERA_PRINCIPAL)].copy()      # universo principal
params = json.loads((META / "parametros.json").read_text(encoding="utf-8"))
REPS = params.get("bootstrap_replicas", 500)
res, prod, un, pm = M.embudo(d)
U = set(prod.index)
claves = {p: M.CLAVES.get(p, M.UNIDAD) for p in pm}
bt = M.boot_embudo(pm, claves, U, reps=REPS)
res["ic90_inf"] = [np.percentile(bt[p], 5) for p in res.paso]
res["ic90_sup"] = [np.percentile(bt[p], 95) for p in res.paso]
res["cambio_pp"] = res.brecha_mediana.diff()
res["nombre"] = [t for _, t, _ in M.PASOS]
res["que_exige"] = [x for _, _, x in M.PASOS]
display(res.round(1))
F = "02_embudo_de_comparabilidad"
for _, r_ in res.iterrows():
    reg(f"paso_{r_.paso}", r_.brecha_mediana, (r_.ic90_inf, r_.ic90_sup), n=r_.productos, fuente=F,
        nota=f"brecha mediana tras: {r_.nombre}")
    reg(f"unidades_{r_.paso}", r_.unidades, fuente=F)
ing, val = res.brecha_mediana.iloc[0], res.brecha_mediana.iloc[-1]
des = (bt["ingenua"] - bt["extremos"]) / bt["ingenua"] * 100
reg("desaparece_pct", (ing - val) / ing * 100, tuple(np.percentile(des, [5, 95])), n=len(prod), fuente=F,
    nota="% de la brecha ingenua que desaparece al validar")
reg("validada", val, (res.ic90_inf.iloc[-1], res.ic90_sup.iloc[-1]), n=len(prod), fuente=F)
reg("validada_promedio", res.brecha_promedio.iloc[-1], fuente=F)

,paso,unidades,productos,brecha_mediana,brecha_promedio,ic90_inf,ic90_sup,cambio_pp,nombre,que_exige
0,ingenua,280,280,25.2,43.8,23.4,28.3,NaN,Comparación ingenua,solo el nombre del producto; promedio de todos los registros del m...
1,presentacion,1304,280,20.6,24.8,18.2,21.1,-4.6,Misma presentación,"mismo tamaño y unidad normalizados (kg, litro, pieza)"
2,marca,1824,280,17.7,23.4,16.6,18.9,-2.8,Misma marca,misma marca y presentación (el mismo artículo)
3,giro,2726,280,17.2,26.9,15.8,18.2,-0.6,Mismo giro,"supermercado contra supermercado, farmacia contra farmacia…"
4,tiendas,1072,280,12.1,15.6,10.9,13.3,-5.1,3 tiendas o más,precio de cada tienda (mediana) y al menos 3 tiendas por municipio
5,periodo,1072,280,12.2,15.6,11.0,13.3,0.1,Mismo periodo,precio ajustado por el día de captura
6,extremos,1072,280,12.2,15.4,11.0,13.3,0.0,Sin errores de captura,sin precios fuera de 0.5×–2× de su propia cadena


## 1. ¿Lo que queda es más que azar?

Aun si todos los municipios tuvieran exactamente los mismos precios, la brecha entre municipios no sería 0: cada
municipio tiene pocas tiendas y de cadenas distintas. **Referencia de azar** = la brecha que resulta de repartir al
azar las mismas tiendas entre los municipios, conservando cuántas tiene cada uno (200 repartos). Si la brecha validada
es parecida a esa referencia, la diferencia «entre municipios» es en realidad diferencia entre tiendas.

In [2]:
fin = un["extremos"][un["extremos"].producto.isin(U)]
sp = M.tiendas_finales(d).merge(fin[M.UNIDAD], on=M.UNIDAD)
az = M.azar(sp, reps=params["repartos_azar"])
az.to_csv(PROC / "azar_unidades.csv", index=False, encoding="utf-8-sig")
pa = az.groupby("producto")[["brecha_pct", "azar_mediana", "azar_p95"]].median()
pa["exceso_pp"] = pa.brecha_pct - pa.azar_mediana
display(pa.describe().round(1))
reg("azar", pa.azar_mediana.median(), n=len(pa), fuente=F, nota="brecha mediana si las tiendas se repartieran al azar entre municipios")
reg("exceso_pp", pa.exceso_pp.median(), n=len(pa), fuente=F, nota="puntos de brecha validada por encima del azar (mediana por nombre de producto)")
reg("pct_productos_sobre_p95", (pa.brecha_pct > pa.azar_p95).mean() * 100, n=len(pa), fuente=F)
prod = prod.join(pa[["azar_mediana", "exceso_pp"]])
prod.reset_index().to_csv(PROC / "brecha_producto_paso.csv", index=False, encoding="utf-8-sig")
res.to_csv(PROC / "embudo.csv", index=False, encoding="utf-8-sig")
queda = max(pa.exceso_pp.median(), 0)
reg("desaparece_azar_pct", (ing - queda) / ing * 100, n=len(pa), fuente=F,
    nota="% de la brecha ingenua que no es comparable o no supera al azar")

,brecha_pct,azar_mediana,azar_p95,exceso_pp
count,280.0,280.0,280.0,280.0
mean,15.5,14.6,19.5,0.9
std,11.1,11.0,13.4,4.9
min,0.5,0.7,1.2,-26.0
25%,7.6,7.5,10.0,-0.7
50%,12.2,11.5,16.3,0.2
75%,18.6,17.5,24.0,2.2
max,64.2,87.7,104.6,23.4


## 2. Cada validación por separado

El orden del embudo atribuye a cada paso lo que cambia después de los anteriores. Para ver cuánto pesa cada validación
sin depender del orden, se aplica **sola** sobre la comparación ingenua.

In [3]:
sep = []
for nombre, clave, tienda in [("Misma presentación", ["producto", "pres_key"], False), ("Misma marca", ["producto", "marca"], False),
                              ("Mismo giro", ["producto", "giro"], False), ("3 tiendas o más", ["producto"], True)]:
    b_ = M.brechas(M.variante(d, clave, tienda), clave)
    s = b_[b_.producto.isin(U)].groupby("producto").brecha_pct.median()
    sep.append({"validación sola": nombre, "nombres": len(s), "brecha mediana": s.median(), "cambio frente a la ingenua (pp)": s.median() - ing})
sep = pd.DataFrame(sep)
display(sep.round(1))
sep.to_csv(PROC / "pasos_por_separado.csv", index=False, encoding="utf-8-sig")
for _, r_ in sep.iterrows():
    reg("sola_" + {"Misma presentación": "presentacion", "Misma marca": "marca", "Mismo giro": "giro", "3 tiendas o más": "tiendas"}[r_["validación sola"]],
        r_["brecha mediana"], n=r_["nombres"], fuente=F)

,validación sola,nombres,brecha mediana,cambio frente a la ingenua (pp)
0,Misma presentación,280,20.6,-4.6
1,Misma marca,280,21.7,-3.4
2,Mismo giro,280,37.5,12.3
3,3 tiendas o más,280,25.8,0.6


## 3. Sensibilidad

Cada variante recalcula el embudo completo con su propio universo de nombres de producto.

In [4]:
def corrida(x, etiqueta, **kw):
    r_, p_, *_ = M.embudo(x, **kw)
    return {"variante": etiqueta, "nombres": len(p_), "ingenua": r_.brecha_mediana.iloc[0], "validada": r_.brecha_mediana.iloc[-1],
            "% que desaparece": (1 - r_.brecha_mediana.iloc[-1] / r_.brecha_mediana.iloc[0]) * 100}
sens = [{"variante": "Principal (3 tiendas, sin errores)", "nombres": len(prod), "ingenua": ing, "validada": val, "% que desaparece": (1 - val / ing) * 100},
        {"variante": "Con todos los precios (incluye posibles errores)", "nombres": len(prod), "ingenua": ing,
         "validada": res.brecha_mediana.iloc[-2], "% que desaparece": (1 - res.brecha_mediana.iloc[-2] / ing) * 100},
        corrida(d, "2 tiendas o más por municipio", min_tiendas=2),
        corrida(d, "5 tiendas o más por municipio", min_tiendas=5),
        corrida(d[d.semana == 1], "Solo semana 1 (17–21 ago)"),
        corrida(d[d.semana == 2], "Solo semana 2 (24–31 ago)"),
        corrida(d_all[d_all.catalogo.isin(M.FUERA_PRINCIPAL)], "Solo medicamentos (sin laboratorio)")]
sens = pd.DataFrame(sens)
display(sens.round(1))
sens.to_csv(PROC / "sensibilidad_embudo.csv", index=False, encoding="utf-8-sig")
for k, i in {"t2": 2, "t5": 3, "sem1": 4, "sem2": 5, "med": 6}.items():
    reg(f"sens_{k}_ingenua", sens.ingenua[i], n=sens.nombres[i], fuente=F)
    reg(f"sens_{k}_validada", sens.validada[i], n=sens.nombres[i], fuente=F)
reg("sens_todos_validada", sens.validada[1], fuente=F)

,variante,nombres,ingenua,validada,% que desaparece
0,"Principal (3 tiendas, sin errores)",280,25.2,12.2,51.7
1,Con todos los precios (incluye posibles errores),280,25.2,12.2,51.7
2,2 tiendas o más por municipio,311,26.3,14.4,45.3
3,5 tiendas o más por municipio,181,23.0,9.7,57.7
4,Solo semana 1 (17–21 ago),248,26.2,13.3,49.0
5,Solo semana 2 (24–31 ago),260,27.5,12.3,55.3
6,Solo medicamentos (sin laboratorio),355,18.1,16.2,10.3


In [5]:
v = entorno.cifras()
display(Markdown(f'''**Conclusión.** Al validar, la diferencia típica entre municipios baja de **{v["paso_ingenua"]:.0f}%** a
**{v["validada"]:.1f}%** (IC90 {v["validada_inf"]:.1f}–{v["validada_sup"]:.1f}%): desaparece {v["desaparece_pct"]:.0f}%
(IC90 {v["desaparece_pct_inf"]:.0f}–{v["desaparece_pct_sup"]:.0f}%). Los pasos que más pesan son la presentación y el
número de tiendas; el periodo y los errores de captura casi no cambian nada. Lo que queda ({v["validada"]:.1f}%) es casi
lo que daría repartir las tiendas al azar ({v["azar"]:.1f}%): en el producto típico, la diferencia propia del municipio
es de {v["exceso_pp"]:.1f} puntos.'''))

**Conclusión.** Al validar, la diferencia típica entre municipios baja de **25%** a
**12.2%** (IC90 11.0–13.3%): desaparece 52%
(IC90 47–59%). Los pasos que más pesan son la presentación y el
número de tiendas; el periodo y los errores de captura casi no cambian nada. Lo que queda (12.2%) es casi
lo que daría repartir las tiendas al azar (11.5%): en el producto típico, la diferencia propia del municipio
es de 0.2 puntos.